In [2]:
import torch
import torch.nn as nn
import sys
sys.path.append("..")
from config.vae_config import VAEConfig
from config.simulation_config import SimulationConfig
from models.encoder import BallEncoder, ObstacleEncoder
from models.decoder import BallDecoder
from models.kalman_filter import KalmanFilter
from models.alphanetwork import AlphaNetwork


class KVAE(nn.Module):
    def __init__(self, cfg: VAEConfig, sim_cfg: SimulationConfig):
        super().__init__()
        self.cfg = cfg

        self.ball_encoder     = BallEncoder(cfg, sim_cfg)
        self.obstacle_encoder = ObstacleEncoder(cfg, sim_cfg)
        self.decoder          = BallDecoder(cfg, sim_cfg)
        self.alpha_net        = AlphaNetwork(cfg)
        self.kalman           = KalmanFilter(cfg)

        # State transition matrices [K, dim_z, dim_z]
        self.A_matrices = nn.Parameter(torch.stack([torch.eye(cfg.dim_z) for _ in range(cfg.num_matrices)]))

        # Observation matrices [K, dim_a, dim_z]
        self.C_matrices = nn.Parameter(cfg.C_std * torch.randn(cfg.num_matrices, cfg.dim_a, cfg.dim_z))

        # Control matrices [K, dim_z, dim_u] — samo ako postoji control input
        if cfg.dim_u > 0:
            self.B_matrices = nn.Parameter(cfg.B_std * torch.randn(cfg.num_matrices, cfg.dim_z, cfg.dim_u))
        else:
            self.B_matrices = None

    def encode(self, ball_seq, obstacle_img):
        """
        Encode ball sequence and obstacle image.
        """
        a_seq, a_mu, a_var = self.ball_encoder(ball_seq)
        h_obs = self.obstacle_encoder(obstacle_img.unsqueeze(1))  # [B, dim_obstacle]
        return a_seq, a_mu, a_var, h_obs

    def decode(self, a_seq):
        """
        Decode latent sequence to ball images.
        """
        return self.decoder(a_seq)

    def forward(self, ball_seq, obstacle_img, u_seq=None, mask=None):
        B, T, H, W = ball_seq.shape

        # Encode
        a_seq, a_mu, a_var = self.ball_encoder(ball_seq)               # a_seq: [B, T, dim_a]
        h_obs = self.obstacle_encoder(obstacle_img.unsqueeze(1))        # [B, dim_obstacle]

        a_var_seq = a_var.view(B, T, self.cfg.dim_a)

        # Kalman filter
        z_filt, P_filt, z_pred, P_pred, a_filt, a_pred, alpha_seq = self.kalman(
            a_seq       = a_seq,
            a_var       = a_var_seq,
            alpha_net   = self.alpha_net,
            h_obs       = h_obs,
            A_matrices  = self.A_matrices,
            C_matrices  = self.C_matrices,
            B_matrices  = self.B_matrices,
            u_seq       = u_seq,
            mask        = mask,
        )

        # Decode
        x_hat_filt = self.decode(a_filt)   # [B, T, H, W]
        x_hat_pred = self.decode(a_pred)   # [B, T, H, W]

        return (
            x_hat_filt, x_hat_pred,
            a_seq, a_mu, a_var,
            z_filt, P_filt, z_pred, P_pred,
            alpha_seq,
        )


if __name__ == "__main__":


    cfg     = VAEConfig()
    sim_cfg = SimulationConfig()
    model   = KVAE(cfg, sim_cfg)

    B, T, H, W = 4, 20, 32, 32
    ball_seq     = torch.zeros(B, T, H, W)
    obstacle_img = torch.zeros(B, H, W)

    out = model(ball_seq, obstacle_img)
    print("x_hat_filt:", out[0].shape)
    print("x_hat_pred:", out[1].shape)
    print("z_filt:    ", out[5].shape)
    print("alpha_seq: ", out[9].shape)

x_hat_filt: torch.Size([4, 20, 32, 32])
x_hat_pred: torch.Size([4, 20, 32, 32])
z_filt:     torch.Size([4, 20, 4])
alpha_seq:  torch.Size([4, 20, 3])
